# Catching Phishing URLs with Logistic Regression
**MLDS × ISACA**

You'll build a model that looks at a URL and estimates the probability that it's a phishing link.


--- 
**How this notebook works**

Cells are marked with these emojis.
- ✏️: Write these. The instructions above each one tell you what to do and what to name things so that later cells work.
- 💬: Stop and think, or talk to the people next to you. Answers can go in the markdown cell below the stop and think cell.


---

## 0. Setup
Run this cell

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

pd.set_option('display.max_colwidth', None)

FILE_PATH = 'phishing_urls.csv'

---

## 1. Load the data
Our data is a sample of the **PhiUSIIL** phishing URL dataset. It has two columns:
- `URL`: the link
- `label`: **1 = phishing**, **0 = legitimate**

✏️ Load the data into a DataFrame called `df` with `pd.read_csv()`, passing it `DATA_URL`. Then check how many rows and columns it has and look at the first few rows.

In [ ]:
# YOUR CODE HERE

---

## 2. Inspect the data

### Class balance
Before modeling, we want to know how many of each class we have. If 99% of URLs were legitimate, a "model" that always says *legit* would be 99% accurate and completely useless.

✏️ Use `.value_counts()` on the `label` column. Then run it again with `normalize=True` to see proportions.

In [ ]:
# YOUR CODE HERE

💬 Is this balanced enough that accuracy is a fair first metric? What accuracy would you get by always guessing the bigger class?

*Your answer:*

### Read some URLs
✏️ Print 10 random **phishing** URLs and 10 random **legitimate** URLs.

**Hint**: Start by filtering the DataFrame using the `label` column. Once you have the URLs for one class, use `.sample()` to randomly select 10 of them. Repeat for the other class.


In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

💬 What do you notice? Which of the red flags that we talked about can you actually spot? Can you spot anything that we didn't discuss?

*Your answer:*

---

## 3. Turn red flags into features
A model can't read a URL. It needs **numbers**. A *feature* is one number we compute from every URL, like its length or whether it contains `@`. Each feature becomes one column.

pandas has string methods that run on a whole column at once through `.str`:
```python
df['URL'].str.len()            # length of every URL
df['URL'].str.count('-')       # number of dashes in every URL
df['URL'].str.contains('@')    # True/False for every URL
```

We'll put all our features inside **one function**, `make_features`. It takes a column of URLs and returns a table of features. Writing it as a function means we can run brand-new URLs through it at the end.

Some red flags only make sense for the **domain** (the `www.example.com` part), so the function also pulls that out as `host`.

### Feature menu

| Red flag (slide 3) | Feature name | What it measures | Hint |
|---|---|---|---|
| Long, cluttered URL | `url_length` | Total characters in the URL | `.str.len()` |
| Long, cluttered URL | `num_digits` | How many digits appear anywhere in the URL | `.str.count(r'\d')` |
| Long, cluttered URL | `num_special` | Count of `?`, `=`, and `&` (query-string clutter) | `.str.count(r'[?=&]')` |
| Long, cluttered URL | `num_hyphens` | Dashes, common in fake domains like `account-verify` | `.str.count('-')` |
| "@" trick | `has_at` | 1 if the URL contains `@`, else 0 | `.str.contains('@').astype(int)` |
| IP address instead of a name | `has_ip` | 1 if the domain is a raw IP address | use `host`: `.str.contains(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}').astype(int)` |
| Real brand buried in a subdomain | `num_subdomains` | Dots in the domain; more dots = more stacked subdomains | use `host`: `.str.count(r'\.')` |
| Lookalike domain | `host_digits` | Digits in the domain (`paypa1`, `micr0soft`) | use `host`: `.str.count(r'\d')` |
| Unusual top-level domain | `suspicious_tld` | 1 if the domain ends in a sketchy TLD | use `host`: `.str.endswith(('.xyz', '.top', '.zip')).astype(int)` |
| *(bonus)* Not encrypted | `is_https` | 1 if the URL starts with `https` | `.str.startswith('https').astype(int)` |

**You don't need all of these.**

💬 **Before you write any code:** which **2** features do you think will be the strongest signal? Write your guess down. We'll check it in section 6.

*My guess:*

✏️ Add **at least 3** features to `make_features`, following the `url_length` example. Each one goes in as a new column of `out`. Use `urls` for whole-URL features and `host` for domain-only features.

In [ ]:
def make_features(urls):
    out = pd.DataFrame(index=urls.index)
    urls = urls.str.replace(r'://www\.', '://', regex=True)   # drop 'www.' (it's just formatting)
    host = urls.str.split('/').str[2].fillna('')              # the domain, e.g. 'paypal.com'

    out['url_length'] = urls.str.len()                        # worked example

    # YOUR CODE HERE

    return out

Now run your function on every URL. The result is our feature table, `X`.

In [ ]:
X = make_features(df['URL'])
y = df['label']
X.head()

### Check
A feature that is almost always 0 doesn't give the model much to learn from. Run this to see how often each feature is non-zero.

In [ ]:
(X > 0).mean().sort_values()

💬 Are any of your features almost never non-zero? Would you keep them? 

---

## 4. Do the features actually differ between classes?
✏️ Compare the **average** of each feature for phishing vs. legitimate URLs.

**Hint**: Think about using `groupby()` with `y`, followed by `.mean()`. `.T` (transposing the data) might make the comparison easier to read.

In [ ]:
# YOUR CODE HERE

✏️ Make a boxplot of `url_length` for each class.

*Hint:* first join the label back on with `X.assign(label=y)`, then make the box plot (`.boxplot()`, filling in `column=` and `by`).

In [ ]:
# YOUR CODE HERE

💬 Which features look the most different between the two classes? Does that match your guess?

*Your answer:*

---

## 5. Train the model

### Split
We train on one part of the data and test on URLs the model has **never seen**, just like a real detector meeting a brand-new phishing site.

✏️ Use `train_test_split` to make `X_train, X_test, y_train, y_test`. Use `test_size=0.2`, `random_state=42`, and `stratify=y`


**Why 42?**: `random_state` is like a Minecraft seed, which will generate the same terrain if you use the same seed. We use `random_state` to split the data exact same way, every time (reproducibility and comparison). We use 42 because it's the Answer to the Ultimate Question of Life, the Universe, and Everything (pop culture reference).  

**Why 0.2?**: A 0.2 (80/20) split balances giving a model enough data to learn stable patterns with keeping a large enough sample for reliable evaluation. In complex workflows, data is often split three ways to include a validation set for tweaking hyperparameters before the final test.

**Why stratify?**: Keeps the ratio of phishing/non-phishing the same in test and train.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

### Scale
`url_length` might be 80 while `has_at` is 0 or 1. If we leave them like that, the weights aren't comparable, so we couldn't say which feature matters most. `StandardScaler` puts every feature on the same scale (mean 0, standard deviation 1).

Note that we **fit** the scaler on the training data only, then apply it to both. The test set has to stay unseen.

In [32]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

### Fit
✏️ Create a `LogisticRegression()` called `model` and `.fit()` it on `X_train_s` and `y_train`.

In [ ]:
# YOUR CODE HERE

---

## 6. Read the weights
Remember from the slides: **z = w₁x₁ + w₂x₂ + … + b**. A big **positive** weight pushes a URL toward *phishing*, and a big **negative** weight pushes it toward *legit*.

In [ ]:
weights = pd.Series(model.coef_[0], index=X.columns).sort_values()
weights

✏️ Make a horizontal bar chart of `weights`.

In [ ]:
# YOUR CODE HERE

💬 Which feature has the biggest positive weight? The biggest negative one? Do they match your guess and ISACA's red flags? Any surprises?

*Your answer:*

---

## 7. How good is it?

### Accuracy vs. a dumb baseline
✏️ Make predictions on the test set and call them `y_pred`. Then compute `accuracy_score(y_test, y_pred)`.

Compare that to the **baseline**: the accuracy you'd get by always guessing the bigger class.

In [ ]:
# YOUR CODE HERE

### Confusion matrix
Rows show what the URL **really** is, and columns show what the model **said**.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Legit', 'Phishing'], cmap='Blues'
)
plt.show()

### Precision and recall
- **Recall**: of all the phishing URLs, how many did we catch?
- **Precision**: of the URLs we flagged, how many were really phishing?

✏️ Compute both with `precision_score()` and `recall_score()`.

In [ ]:
# YOUR CODE HERE

💬 Which mistake is worse here: a missed phishing link (false negative) or a false alarm (false positive)? Which box in the confusion matrix is that?

*Your answer:*

---

## 8. Move the threshold
`model.predict` calls anything with P(phishing) > 0.5 phishing. That 0.5 is a choice, not a law.

`predict_proba` gives us the actual probabilities, so we can pick our own cutoff.

In [ ]:
probs = model.predict_proba(X_test_s)[:, 1]   # P(phishing) for each test URL
probs[:10].round(3)

✏️ Fill in the loop: for each threshold, turn `probs` into 0/1 predictions (`probs >= t`), then compute precision and recall.

In [ ]:
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    preds_t = # YOUR CODE HERE
    rows.append({
        'threshold': t,
        'precision': # YOUR CODE HERE,
        'recall':    # YOUR CODE HERE,  
    })

pd.DataFrame(rows).round(3)

,threshold,precision,recall
0,0.2,0.921,0.951
1,0.3,0.958,0.937
2,0.4,0.968,0.925
3,0.5,0.982,0.918
4,0.6,0.987,0.910
5,0.7,0.993,0.894
6,0.8,0.997,0.878


💬 If you ran the security team, where would you set the threshold? What would you be trading away?

*Your answer:*

---

## 9. What did it miss?
These are phishing URLs that the model called **safe** (false negatives).

In [ ]:
test_results = pd.DataFrame({
    'URL': df.loc[X_test.index, 'URL'],
    'actual': y_test,
    'predicted': y_pred,
    'p_phishing': probs.round(3),
})

missed = test_results[(test_results['actual'] == 1) & (test_results['predicted'] == 0)]
print(len(missed), 'phishing URLs missed')
missed.sample(min(10, len(missed)), random_state=0)

💬 What do the missed ones have in common? Which red flag would have caught them, if any?

*Your answer:*

### 🏆 Challenge: fool the model
✏️ Write URLs that a person would flag as phishing but the model thinks are safe. Add them to the list and run the cell. Can you get one under 0.5?

In [ ]:
my_urls = pd.Series([
    'https://www.paypal.com',            # a real one, for comparison
    # YOUR CODE HERE
])

my_probs = model.predict_proba(scaler.transform(make_features(my_urls)))[:, 1]
pd.DataFrame({'URL': my_urls, 'p_phishing': my_probs.round(3)})

---

## 10. Wrap-up
- A handful of string features plus logistic regression already gets a working detector, and you can **read** why it decides what it does.
- In real systems, this would be one layer: a **blocklist** first, a fast model like this one as a first pass, then heavier models (random forests, XGBoost, BERT-style models) on anything suspicious.
- Attackers adapt. Once they know which features get flagged, they avoid them, so real detectors are constantly retrained and combined with other signals.

💬 **If you had another hour,** what feature would you add next?

*Your answer:*